# SylloGym — GRPO Training with Unsloth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/syllogym/blob/main/training/train_grpo.ipynb)

This notebook trains a language model to perform **legal syllogistic reasoning** using **GRPO** (Group Relative Policy Optimization).

The model learns to:
1. Apply a legal rule (major premise) to case facts (minor premise)
2. Derive the correct legal conclusion
3. Produce structured reasoning with `<reasoning>` and `<answer>` tags

**Environment:** [SylloGym](https://huggingface.co/spaces/openenv/syllogym_env) — built on [LegalBench](https://huggingface.co/datasets/nguha/legalbench)  
**Model:** Qwen2.5-3B-Instruct (via Unsloth)  
**Training:** TRL GRPOTrainer + Unsloth LoRA  
**Compute:** Google Colab A100 (or T4 with smaller model)

---

## Reward Structure

| Component | Value | Condition |
|-----------|-------|----------|
| Format reward | +0.1 | `<reasoning>` + `<answer>` tags present |
| Answer reward | +1.0 | Exact match with ground truth |
| Reasoning quality | +0.2 | Reasoning references rule + facts keywords |
| **Max total** | **1.3** | |

In [ ]:
# ============================================================
# Setup — clone repo and install dependencies
# ============================================================

import os, sys

REPO_NAME = "syllogym"
REPO_URL = "https://github.com/eliot-gtn/syllogym.git"

# Clone the repo if not already present
if not os.path.exists(f"/content/{REPO_NAME}"):
    !git clone -b develop {REPO_URL} /content/{REPO_NAME}
else:
    !git -C /content/{REPO_NAME} pull

# Add envs/ to Python path so syllogym_env can be imported
sys.path.insert(0, f"/content/{REPO_NAME}/envs")

# Install dependencies
%pip install -q "unsloth[colab-new]" "trl>=0.15.2" trackio
%pip install -q openenv-core "datasets>=2.19.0" huggingface_hub

print("Setup complete.")

In [ ]:
# ============================================================
# 1. Load model with Unsloth
# ============================================================

from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Change to compare models
# MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # Smaller, faster
# MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"  # Llama comparison

MAX_SEQ_LENGTH = 2048
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    max_seq_length=MAX_SEQ_LENGTH,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f'Model loaded: {MODEL_NAME}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')

In [ ]:
# ============================================================
# 2. Connect to SylloGym environment
# ============================================================

from syllogym_env import SylloGymEnv, SylloAction

# HuggingFace Space (after deployment)
SYLLOGYM_URL = "https://openenv-syllogym-env.hf.space"
# For local testing:
# SYLLOGYM_URL = "http://localhost:8000"

env = SylloGymEnv(base_url=SYLLOGYM_URL)

# Quick sanity check
result = env.reset()
obs = result.observation
print(f'Environment connected!')
print(f'Task: {obs.task_name} (difficulty {obs.difficulty})')
print(f'Rule (first 100 chars): {obs.rule[:100]}...')
print(f'Facts (first 100 chars): {obs.facts[:100]}...')
print(f'Valid answers: {obs.valid_answers}')

In [ ]:
# ============================================================
# 3. System prompt and dataset construction
# ============================================================

from datasets import Dataset

SYSTEM_PROMPT = """You are a legal reasoning expert. You will be given:
1. A legal RULE
2. FACTS of a case
3. A QUESTION

Apply the rule to the facts using deductive reasoning (syllogism):
- Major premise: the rule
- Minor premise: the facts
- Conclusion: your answer

Respond in this exact format:
<reasoning>
[Your step-by-step analysis applying the rule to the facts. Be concise: 2-4 sentences.]
</reasoning>
<answer>[Your answer here]</answer>

Only include your answer word/phrase inside the <answer> tags (no punctuation or explanation)."""


def make_prompt_from_obs(obs) -> list[dict]:
    """Build the chat messages for a given observation."""
    valid = " | ".join(obs.valid_answers)
    user_content = (
        f"RULE:\n{obs.rule}\n\n"
        f"FACTS:\n{obs.facts}\n\n"
        f"QUESTION: {obs.question}\n\n"
        f"Valid answers: {valid}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


# Pre-sample problems from the env to build a static dataset
# GRPOTrainer will call reward_funcs on each generated completion,
# but we store the problem context (obs) in dataset extras.
print('Sampling problems from environment...')

DATASET_SIZE = 2000
problems = []

for i in range(DATASET_SIZE):
    result = env.reset()
    obs = result.observation
    if obs.facts and not obs.facts.startswith('[Dataset'):
        problems.append({
            "prompt": make_prompt_from_obs(obs),
            "task_name": obs.task_name,
            "difficulty": obs.difficulty,
            "correct_answer": obs.correct_answer,
            "valid_answers": obs.valid_answers,
            "rule": obs.rule,
            "facts": obs.facts,
        })

dataset = Dataset.from_list(problems)
print(f'Dataset built: {len(dataset)} problems')

# Distribution over tasks
from collections import Counter
task_counts = Counter(p["task_name"] for p in problems)
for task, count in sorted(task_counts.items()):
    print(f'  {task:<30s}: {count}')

In [ ]:
# ============================================================
# 4. Reward functions
# ============================================================

import re


def reward_format(completions, **kwargs):
    """
    Reward 0.1 if the model uses <reasoning>...</reasoning> and <answer>...</answer> tags.
    Encourages structured output format.
    """
    scores = []
    for completion in completions:
        response = completion[0]["content"] if isinstance(completion, list) else completion
        has_reasoning = bool(re.search(r"<reasoning>.*?</reasoning>", response, re.DOTALL))
        has_answer = bool(re.search(r"<answer>.*?</answer>", response, re.DOTALL | re.IGNORECASE))
        scores.append(0.1 if (has_reasoning and has_answer) else 0.0)
    return scores


def reward_answer(completions, correct_answer=None, valid_answers=None, **kwargs):
    """
    Reward 1.0 for correct answer, 0.0 otherwise.
    Extracts answer from <answer> tags if present, otherwise uses the full response.
    """
    scores = []
    # correct_answer is passed as a list (one per sample in batch)
    if not isinstance(correct_answer, list):
        correct_answer = [correct_answer] * len(completions)
    
    for i, completion in enumerate(completions):
        response = completion[0]["content"] if isinstance(completion, list) else completion
        gt = str(correct_answer[i]).strip().lower() if correct_answer[i] else ""
        
        # Try to extract from <answer> tags
        tag_match = re.search(r"<answer>(.*?)</answer>", response, re.DOTALL | re.IGNORECASE)
        if tag_match:
            predicted = tag_match.group(1).strip().lower()
        else:
            predicted = response.strip().lower()
        
        scores.append(1.0 if predicted == gt else 0.0)
    return scores


def reward_reasoning_quality(completions, rule=None, facts=None, **kwargs):
    """
    Reward 0.2 if the reasoning references both the rule and the facts.
    Encourages genuine deductive reasoning rather than pattern matching.
    """
    STOPWORDS = {"under", "shall", "must", "with", "that", "this", "from",
                 "have", "been", "were", "they", "their", "there", "when", "which"}

    scores = []
    if not isinstance(rule, list):
        rule = [rule] * len(completions)
    if not isinstance(facts, list):
        facts = [facts] * len(completions)
    
    for i, completion in enumerate(completions):
        response = completion[0]["content"] if isinstance(completion, list) else completion
        
        # Extract reasoning content
        reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", response, re.DOTALL)
        reasoning = reasoning_match.group(1).lower() if reasoning_match else response.lower()
        
        rule_i = str(rule[i] or "").lower()
        facts_i = str(facts[i] or "").lower()
        
        rule_words = {w for w in re.findall(r"\b[a-z]{5,}\b", rule_i) if w not in STOPWORDS}
        facts_words = {w for w in re.findall(r"\b[a-z]{5,}\b", facts_i) if w not in STOPWORDS}
        
        rule_hits = sum(1 for w in rule_words if w in reasoning)
        facts_hits = sum(1 for w in facts_words if w in reasoning)
        
        scores.append(0.2 if (rule_hits >= 2 and facts_hits >= 2) else 0.0)
    return scores


print('Reward functions defined:')
print('  reward_format         : +0.1 for <reasoning>+<answer> tags')
print('  reward_answer         : +1.0 for correct answer')
print('  reward_reasoning_quality: +0.2 for referencing rule+facts')
print('  Max total reward      : 1.3')

In [ ]:
# ============================================================
# 5. GRPO Training Configuration
# ============================================================

from trl import GRPOConfig, GRPOTrainer

# Compute max prompt length
sample_prompt = tokenizer.apply_chat_template(
    make_prompt_from_obs(obs),
    tokenize=False,
    add_generation_prompt=True,
)
max_prompt_length = len(tokenizer(sample_prompt)["input_ids"]) + 32  # small buffer
max_completion_length = MAX_SEQ_LENGTH - max_prompt_length

print(f'Max prompt length: {max_prompt_length}')
print(f'Max completion length: {max_completion_length}')

HF_USERNAME = "YOUR_HF_USERNAME"  # <-- Set your HuggingFace username
OUTPUT_DIR = f"syllogym-grpo-{MODEL_NAME.split('/')[-1]}"

training_args = GRPOConfig(
    # Core GRPO params
    num_generations=4,              # Generate 4 candidates per prompt for group advantage
    temperature=0.9,                # Exploration temperature
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,

    # Optimization
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",

    # Batch settings
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # Effective batch = 4 * num_generations = 16

    # Training duration
    max_steps=500,
    save_steps=100,

    # Logging
    logging_steps=1,
    report_to="trackio",

    output_dir=OUTPUT_DIR,
)

print(f'Training config: {training_args.max_steps} steps, lr={training_args.learning_rate}')

In [ ]:
# ============================================================
# 6. Create trainer and train
# ============================================================

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        reward_format,
        reward_answer,
        reward_reasoning_quality,
    ],
    args=training_args,
    train_dataset=dataset,
)

# Log GPU memory before training
if torch.cuda.is_available():
    start_mem = torch.cuda.max_memory_reserved() / 1e9
    max_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU memory before training: {start_mem:.1f} / {max_mem:.1f} GB')

# Train!
trainer_stats = trainer.train()

# Log GPU memory after training
if torch.cuda.is_available():
    used_mem = torch.cuda.max_memory_reserved() / 1e9
    print(f'\nPeak GPU memory: {used_mem:.1f} / {max_mem:.1f} GB ({used_mem/max_mem:.0%})')
    print(f'Training time: {trainer_stats.metrics["train_runtime"] / 60:.1f} minutes')

In [ ]:
# ============================================================
# 7. Quick evaluation on a few examples
# ============================================================

print('Testing trained model on a few examples...\n')

FastLanguageModel.for_inference(model)  # Enable fast inference mode

from transformers import TextStreamer

for task_name in ["diversity_1", "ucc_v_common_law", "hearsay"]:
    result = env.reset(task_mode="single", task_name=task_name)
    obs = result.observation
    
    if obs.facts.startswith('[Dataset'):
        continue
    
    messages = make_prompt_from_obs(obs)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    print(f'Task: {task_name} | Correct: {obs.correct_answer}')
    print('Model response:')
    _ = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.1,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print('\n' + '='*60 + '\n')

In [ ]:
# ============================================================
# 8. Save model to HuggingFace Hub
# ============================================================

# Login to HuggingFace first:
# from huggingface_hub import notebook_login
# notebook_login()

REPO_ID = f"{HF_USERNAME}/{OUTPUT_DIR}"

# Save merged 16-bit model
if False:  # Set to True to push
    model.push_to_hub_merged(
        REPO_ID,
        tokenizer,
        save_method="merged_16bit",
        token="hf_...",  # Your HF token
    )
    print(f'Model pushed to: https://huggingface.co/{REPO_ID}')

# Save adapter only (smaller, can be merged later)
if False:  # Set to True to push adapter
    model.push_to_hub_merged(
        REPO_ID + "-adapter",
        tokenizer,
        save_method="lora",
        token="hf_...",
    )

env.close()
print('Training complete!')

## Next Steps

1. Run `evaluation/eval_baseline.ipynb` with the trained model to measure improvement
2. Compare against the baseline (pre-training) results
3. Try different models: `Qwen2.5-1.5B-Instruct`, `Llama-3.2-3B-Instruct`
4. Experiment with curriculum mode by setting `task_mode="curriculum"` in the env

## Training Tips

- Watch the `reward/reward` metric in TrackIO — it should increase over training
- If reward is stuck at 0 for 100+ steps, the model may need more warmup or lower learning rate
- `reward_format` should reach ~1.0 quickly (easy); `reward_answer` is the hard signal
- For T4 GPU: reduce `LORA_RANK=4`, `num_generations=2`, `max_seq_length=1024`